# 07. Denmark Marker cabbage test supplement

[Notebook 01](01_external_reference_audit.ipynb) introduces the external sources and their roles. The 2023 Marker declarations add a cabbage-only external check outside the French development source.

Saved outputs may describe an earlier run. [Notebook 08](08_dataset_summary.ipynb) shows the parcels used in the saved experiment.

## 1. Settings for this run

Review the settings before running the notebook from top to bottom.

| Setting | What it controls |
|---|---|
| `run_satellite_collection` | Enables Earth Engine submission and retrieval, subject to the notebook's reference mode. |
| `submit_only` | Queues cloud work for retrieval in a later run. |
| `parcels_per_batch` | Candidate parcel limit for each new batch. |
| `extra_batches` | Highest numbered extra slot to consider in the current round, after the regular batch. |

A **campaign** holds the collection goal and its records, a **round** groups batch plans, and a **batch** is a set of selected parcels with an imagery plan. Rerunning with the same extra-slot limit revisits the current round and can resume unfinished work. For example, changing 25 to 26 makes slot 0026 available in that round if eligible parcels remain. [Collection guide](README.md)

In [ ]:
# Edit these values before running collection.
run_satellite_collection = True
submit_only = False
parcels_per_batch = 50
extra_batches = 1


In [ ]:
from pathlib import Path
import sys

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))
from crop_pipeline import notebook_setup
from crop_pipeline.preprocessing_workflow import paths
from crop_pipeline.sources import adaptive, annual, denmark, supplemental

settings_path = root / "settings.local.json"
settings = notebook_setup.ensure_local_settings(settings_path)
_, report, _ = paths(root)
archive = annual.archive_root(settings, report, repository_root=root)
from crop_pipeline.notebook_display import show_table, show_mapping

## 2. Obtain references

Load the Danish reference snapshot. The loader obtains it when missing. The declarations link crop codes to mapped field polygons.

In [ ]:
references = root / "data/external/denmark_marker/cabbage_2023.geojson"
snapshot = denmark.load_cabbage(references)

## 3. Check parcel eligibility

Count declarations matching the accepted white/red cabbage code set (408 and 418), then count distinct valid polygons. The next step screens the footprints for overlap.

In [ ]:
features = snapshot["features"]
show_mapping({"Declared cabbage records": snapshot["matching_records"],
              "Unique valid cabbage polygons": len(features)}, title="Denmark Marker reference pool")

## 4. Plan or resume batches

The planner uses eligible field footprints to create imagery requests. Saved plans carry their original selections and project splits. The batch table shows selected parcels and planned tiles for plans returned in this run.

In [ ]:
batches = supplemental.prepare(
    "denmark_marker", report, archive, features, references,
    limit=parcels_per_batch, extra_batches=extra_batches,
)

# Count parcels and imagery tiles separately.
show_table([
    {"Batch": Path(context["folder"]).name, "Selected parcels": context["candidate_parcels"],
      "Planned tiles": plan["estimate"]["tiles"]}
    for plan, context in batches if plan is not None
], title="Parcel and imagery workload by batch")

## 5. Collect annual imagery

Earth Engine uses the planned field footprints and years to find and process Sentinel-2 imagery. With `submit_only = True`, it can continue processing after submission finishes and the laptop is closed. On a later run, `submit_only = False` retrieves completed results, caches selected-pixel tables as Parquet and assembles annual TIFFs. Matching verified work can be reused. [Transport and caching](../../docs/reference/transport.md)

In [ ]:
project = notebook_setup.project_for_collection(settings, settings_path, required=run_satellite_collection)
result = adaptive.collect_batches(
    batches, project, enabled=run_satellite_collection, authenticate=run_satellite_collection,
    submit_only=submit_only, download_workers=2, intervals_per_request=6,
)


## 6. Review collection status

Use the batch status to see which work finished and which can resume later. Verify the parcels and windows that pass imagery checks through preprocessing.
When retrieval is enabled, this notebook also reports the Denmark green-up window check.

In [ ]:
# Status comes from this run's collection response.
show_mapping({"Collection status": result["status"].replace("_", " ").title()},
             title="Collection result")
show_table([
    {"Batch": entry["batch"], "Status": entry["result"]["status"].replace("_", " ").title()}
    for entry in result.get("batches", [])
], title="Collection status by batch")
if run_satellite_collection and not submit_only:
    import csv
    from crop_pipeline import cabbage_test_check

    check_folder = cabbage_test_check.run(root)
    with (check_folder / "window_counts.csv").open() as handle:
        counts = next(row for row in csv.DictReader(handle)
                      if row["test_cohort"] == "denmark_marker_cabbage_2023_v1"
                      and row["window"] == "aligned" and row["population"] == "full")
    print("Denmark usable green-up parcels:", counts["usable_parcels"])
    print("Denmark usable green-up pixels:", counts["usable_pixels"])
    print("Audit:", check_folder)

## Saved outputs

Browse reference files, batch states and imagery through the organized collection index.

In [ ]:
from crop_pipeline.output_layout import refresh

output_layout = refresh(root)
print("Organized collection outputs:", root / "pipeline_outputs/collection")

Continue to [Notebook 08](08_dataset_summary.ipynb) for the combined dataset profile and [Preprocessing 01](../preprocessing/01_imagery_quality.ipynb) for imagery quality and coverage.